# 06 - Federated Learning Training (Round Robin)

## Overview

Train FL models using **Round Robin** strategy with incremental XGBoost/LightGBM.
Each round: 1 client trains + evaluates. Global model accumulates trees round by round.

**NOTE**: Set `TRAIN_SUBSAMPLE_RATIO = 0.25` in `src/config.py` for 25% data (faster).
Set to `1.0` for full training once hyperparameters are settled.

In [ ]:
# Imports
import numpy as np
import pandas as pd
import os
import json
import time
import subprocess
import warnings
warnings.filterwarnings('ignore')

import sys
sys.path.insert(0, '/home/willtanoe/Documents/fl-xgb-thesis')
from src.config import (
    NUM_CLIENTS, FL_ROUNDS, LOCAL_EPOCHS, FRACTION_FIT,
    XGB_PARAMS, LGB_PARAMS, SEED
)

import flwr as fl

import joblib
from sklearn.metrics import f1_score, accuracy_score, classification_report
from tqdm import tqdm

# Paths
PREPROCESSED_DIR = "/home/willtanoe/Documents/fl-xgb-thesis/results/preprocessed"
CLIENTS_DIR = "/home/willtanoe/Documents/fl-xgb-thesis/results/clients"
MODELS_DIR = "/home/willtanoe/Documents/fl-xgb-thesis/results/models"
LOGS_DIR = "/home/willtanoe/Documents/fl-xgb-thesis/results/logs"

os.makedirs(MODELS_DIR, exist_ok=True)

# Load metadata
with open(os.path.join(PREPROCESSED_DIR, "metadata.json"), 'r') as f:
    metadata = json.load(f)

feature_cols = metadata['feature_cols']
num_classes = metadata['num_classes']

print(f"FL Training Configuration:")
print(f"  Num clients: {NUM_CLIENTS}")
print(f"  FL rounds: {FL_ROUNDS}")
print(f"  Local epochs: {LOCAL_EPOCHS}")
print(f"  Fraction fit: {FRACTION_FIT}")
print(f"  Num classes: {num_classes}")

## 1. Define Flower Strategy

In [ ]:
print("Strategy defined in flwr_app/server_app.py via server_fn()")

## 2. Load Validation Data for Centralized Evaluation

In [ ]:
# Load validation and test sets
print("Loading validation and test sets...")
val_df = pd.read_parquet(os.path.join(PREPROCESSED_DIR, "val.parquet"))
test_df = pd.read_parquet(os.path.join(PREPROCESSED_DIR, "test.parquet"))

X_val = val_df[feature_cols].values
y_val = val_df['label'].values
X_test = test_df[feature_cols].values
y_test = test_df['label'].values

print(f"Val: {X_val.shape}, Test: {X_test.shape}")

# Load label encoder for class names
le = joblib.load(os.path.join(PREPROCESSED_DIR, "label_encoder.pkl"))

In [ ]:
# Save a 10k sample of val set for Flower App (avoids loading 1.1M rows per actor)
val_sample = val_df.sample(n=min(10000, len(val_df)), random_state=42)
val_sample.to_parquet(os.path.join(PREPROCESSED_DIR, "val_sample.parquet"))
print(f"Saved val_sample.parquet ({len(val_sample)} rows) for Flower App eval")

## 3. XGBoost FL Training

In [ ]:
PROJECT_DIR = "/home/willtanoe/Documents/fl-xgb-thesis"
FLWR_APP = os.path.join(PROJECT_DIR, "flwr_app")
LOGS_DIR = os.path.join(PROJECT_DIR, "results", "logs")
import subprocess as sp
xgb_log_path = os.path.join(LOGS_DIR, "xgb_flwr_run.log")
os.makedirs(LOGS_DIR, exist_ok=True)

print("Running XGBoost FL training via flwr run...")
print(f"Clients: {NUM_CLIENTS}, Rounds: {FL_ROUNDS}\n")
print(f"Logging to {xgb_log_path}\n")

with open(xgb_log_path, "w") as log_f:
    proc = sp.Popen(
        ["flwr", "run", FLWR_APP, "--stream",
         "--federation-config",
         "client-resources-num-cpus=1 num-supernodes=5"],
        stdout=sp.PIPE, stderr=sp.STDOUT,
        cwd=PROJECT_DIR,
        bufsize=1, universal_newlines=True,
    )
    for line in proc.stdout:
        print(line, end="")
        log_f.write(line)
    proc.wait()

print("\nXGBoost FL training complete.")


## 4. LightGBM FL Training

In [ ]:
print("Running LightGBM FL training via flwr run...")
print(f"Clients: {NUM_CLIENTS}, Rounds: {FL_ROUNDS}\n")
lgb_log_path = os.path.join(LOGS_DIR, "lgb_flwr_run.log")
print(f"Logging to {lgb_log_path}\n")

with open(lgb_log_path, "w") as log_f:
    proc = sp.Popen(
        ["flwr", "run", FLWR_APP, "--stream",
         "--federation-config",
         "client-resources-num-cpus=1 num-supernodes=5",
         "--run-config", 'model_type="lgb"'],
        stdout=sp.PIPE, stderr=sp.STDOUT,
        cwd=PROJECT_DIR,
        bufsize=1, universal_newlines=True,
    )
    for line in proc.stdout:
        print(line, end="")
        log_f.write(line)
    proc.wait()

print("\nLightGBM FL training complete.")


## 5. Save Training History

In [ ]:
# Load metrics from flwr run output files
xgb_metrics_path = os.path.join(LOGS_DIR, "fl_history_xgb.json")
lgb_metrics_path = os.path.join(LOGS_DIR, "fl_history_lgb.json")

xgb_data = {}
lgb_data = {}

if os.path.exists(xgb_metrics_path):
    with open(xgb_metrics_path) as f:
        xgb_data = json.load(f)
    print(f"Loaded XGBoost metrics: {len(xgb_data.get('rounds', []))} rounds")

if os.path.exists(lgb_metrics_path):
    with open(lgb_metrics_path) as f:
        lgb_data = json.load(f)
    print(f"Loaded LightGBM metrics: {len(lgb_data.get('rounds', []))} rounds")

# Build history objects for backward compat with plotting code
class MockHistory:
    def __init__(self, data):
        self.metrics_distributed = {}
        if data.get("f1_macro"):
            rounds = data.get("rounds", [])
            self.metrics_distributed["f1_macro"] = list(zip(rounds, data["f1_macro"]))
        self.losses_distributed = [(r, 0.0) for r in data.get("rounds", [])]

history = MockHistory(xgb_data)
history_lgb = MockHistory(lgb_data)

# Save combined history (backwards compat)
history_data = {
    'xgb': {
        'rounds': xgb_data.get('rounds', []),
        'f1_macro': xgb_data.get('f1_macro', []),
        'accuracy': xgb_data.get('accuracy', []),
    },
    'lgb': {
        'rounds': lgb_data.get('rounds', []),
        'f1_macro': lgb_data.get('f1_macro', []),
        'accuracy': lgb_data.get('accuracy', []),
    },
    'config': {
        'num_clients': NUM_CLIENTS,
        'fl_rounds': FL_ROUNDS,
        'num_classes': num_classes,
    }
}

with open(os.path.join(LOGS_DIR, 'fl_training_history.json'), 'w') as f:
    json.dump(history_data, f, indent=2)
print("Training history saved: results/logs/fl_training_history.json")

## 6. Plot Training Curves

In [ ]:
import matplotlib.pyplot as plt

# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# XGBoost
ax = axes[0]
if xgb_data.get('f1_macro'):
    ax.plot(xgb_data['rounds'], xgb_data['f1_macro'], 'b-', label='XGBoost F1 Macro')
    ax.plot(xgb_data['rounds'], xgb_data['accuracy'], 'b--', label='XGBoost Accuracy')
ax.set_xlabel('FL Round')
ax.set_ylabel('Score')
ax.set_title('XGBoost Training')
ax.legend()
ax.grid(True, alpha=0.3)

# LightGBM
ax = axes[1]
if lgb_data.get('f1_macro'):
    ax.plot(lgb_data['rounds'], lgb_data['f1_macro'], 'orange', label='LightGBM F1 Macro')
    ax.plot(lgb_data['rounds'], lgb_data['accuracy'], 'orange', ls='--', label='LightGBM Accuracy')
ax.set_xlabel('FL Round')
ax.set_ylabel('Score')
ax.set_title('LightGBM Training')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(LOGS_DIR, '06_fl_training_curves.png'), dpi=150, bbox_inches='tight')
plt.show()

print("Figure saved: results/logs/06_fl_training_curves.png")

## Summary

- Trained XGBoost with FedAvg for {FL_ROUNDS} rounds
- Trained LightGBM with FedAvg for {FL_ROUNDS} rounds
- Client/Server logic defined in `flwr_app/` (ClientApp + ServerApp)
- Training history saved to `results/logs/fl_training_history.json`

**Next**: Run `07_evaluation.ipynb` for full evaluation on test set.